# Regime-Dependent Risk & Performance Attribution

**Research question:** Do full-sample risk metrics misrepresent asset risk profiles across market regimes?

**Thesis:** Yes. CAPM beta, volatility, VaR, and Sharpe ratio all vary materially across regimes. Full-sample averages are statistical artefacts that mask the true risk profile of every asset examined.

| | |
|---|---|
| **Assets** | NVDA, JPM, PG (equity); SPY (benchmark) |
| **Period** | 2019–2024 (1 000 trading days) |
| **Classifier** | SPY 30-day rolling vol split at in-sample median → Low-Vol / High-Vol |
| **Pipeline** | Regime detection → per-regime CAPM (HC3) → risk metrics → GARCH VaR → stress testing |

---

## 1. Setup

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import TICKERS, RF_ANNUAL, setup_style
from src import data, regimes, metrics, capm, volatility, backtest, stress, plots

setup_style()

---
## 2. Data Loading

In [2]:
prices = data.load_prices()
print(f"Loaded {len(prices)} trading days: "
      f"{prices.index[0].date()} → {prices.index[-1].date()}")
prices.describe().round(2)

Loaded 1258 trading days: 2019-01-02 → 2023-12-29


,NVDA,JPM,PG,SPY
count,1258.00,1258.00,1258.00,1258.00
mean,17.67,113.80,117.63,349.78
std,12.44,22.12,17.89,65.10
min,3.17,66.13,73.78,203.91
25%,6.79,93.16,103.01,284.56
50%,14.51,115.04,119.57,366.70
75%,23.00,133.26,133.01,405.98
max,50.26,159.80,145.24,461.58


---
## 3. Log Returns & Distributional Diagnostics

Log returns are time-additive and symmetric, making them suitable for compounding.
We confirm fat tails before any risk modelling — Gaussian VaR will systematically
underestimate tail risk if this assumption is violated.

In [3]:
rets = data.log_returns(prices)
diag = data.return_diagnostics(rets)

print(f"{'Ticker':<6} {'Excess Kurt':>12} {'Skew':>8} {'JB p-val':>12} {'ADF p-val':>12}")
print('-' * 52)
for t, row in diag.iterrows():
    print(f"{t:<6} {row['kurt']:>12.3f} {row['skew']:>8.3f} {row['jb_p']:>12.2e} {row['adf_p']:>12.2e}")

print("\nAll JB p-values ≈ 0 → normality rejected.  All ADF p-values ≈ 0 → returns stationary.")

Ticker  Excess Kurt     Skew     JB p-val    ADF p-val
----------------------------------------------------
NVDA          4.045    0.051    6.91e-187     9.71e-20
JPM          12.887   -0.052     0.00e+00     1.09e-17
PG           11.281   -0.102     0.00e+00     3.18e-20
SPY          12.363   -0.820     0.00e+00     9.05e-19

All JB p-values ≈ 0 → normality rejected.  All ADF p-values ≈ 0 → returns stationary.


In [4]:
fig = plots.plot_qq(rets, diag)
fig.savefig('../data/qq_plots.png', dpi=150, bbox_inches='tight')

---
## 4. Regime Classification

SPY 30-day rolling volatility vs its full-sample median → binary Low-Vol / High-Vol labels.
Median split ensures balanced (~50/50) classes for reliable per-regime estimation.

> **Look-ahead note:** the full-sample median is acceptable for retrospective attribution.
> A live system must use an expanding-window median to avoid forward-looking bias.

In [5]:
regime, rets_a, threshold = regimes.classify_regimes(rets)

vc = regime.value_counts()
for r, n in vc.items():
    print(f"  {r:<10}: {n:>4} days ({100*n/vc.sum():.1f}%)")
print(f"  Threshold  : {threshold:.5f} (daily σ)")

  High-Vol  :  614 days (50.0%)
  Low-Vol   :  614 days (50.0%)
  Threshold  : 0.00927 (daily σ)


In [6]:
# Point-in-time check: the same split with an expanding median (labels use data through t only)
regime_x, rets_x, thr_x = regimes.classify_regimes(rets, method='expanding')
common = regime.index.intersection(regime_x.index)
agree = (regime.loc[common] == regime_x.loc[common]).mean()
print(f"Expanding-median regimes from {regime_x.index[0].date()}: agree with full-sample labels on {agree:.1%} of {len(common)} common days")

rows_x = []
for t in TICKERS:
    for lbl in ['Low-Vol', 'High-Vol']:
        for name, reg, ra in [('full_sample', regime, rets_a), ('expanding', regime_x, rets_x)]:
            m = metrics.calc_metrics(ra.loc[reg == lbl, t], lbl)
            rows_x.append({'Ticker': t, 'Regime': lbl, 'Method': name, 'Sharpe': m['Sharpe'], 'CVaR95': m['CVaR95']})
regime_method_df = pd.DataFrame(rows_x)
print(regime_method_df.pivot_table(index=['Ticker', 'Regime'], columns='Method', values='Sharpe').round(3).to_string())


Expanding-median regimes from 2020-02-13: agree with full-sample labels on 92.4% of 977 common days
Method           expanding  full_sample
Ticker Regime                          
JPM    High-Vol     -0.139        0.026
       Low-Vol       1.336        1.213
NVDA   High-Vol      0.320        0.683
       Low-Vol       3.821        2.415
PG     High-Vol      0.212        0.314
       Low-Vol       0.210        0.599
SPY    High-Vol      0.099        0.201
       Low-Vol       1.541        1.467


In [7]:
spy_rvol = rets['SPY'].rolling(30).std()
fig = plots.plot_regimes(prices, regime, spy_rvol, threshold)
fig.savefig('../data/regime_chart.png', dpi=150, bbox_inches='tight')

---
## 5. Modelling
### 5a. Regime-Conditional Risk Metrics

In [8]:
rows = []
for t in TICKERS:
    for lbl in ['Low-Vol', 'High-Vol', 'Full Sample']:
        r = rets_a[t] if lbl == 'Full Sample' else rets_a.loc[regime == lbl, t]
        m = metrics.calc_metrics(r, lbl)
        m['Ticker'] = t
        rows.append(m)

metrics_df = pd.DataFrame(rows)
cols = ['Label', 'N', 'AnnRet', 'AnnVol', 'Sharpe', 'Sortino', 'MaxDD', 'Calmar', 'VaR95', 'CVaR95']

for t in TICKERS:
    sub = metrics_df[metrics_df.Ticker == t][cols]
    print(f'\n{"="*90}\n  {t}  (RF={RF_ANNUAL*100:.1f}%)\n{"="*90}')
    print(sub.to_string(index=False, float_format=lambda x: f'{x:8.3f}'))

    lv = sub[sub.Label == 'Low-Vol'].iloc[0]
    hv = sub[sub.Label == 'High-Vol'].iloc[0]
    fs = sub[sub.Label == 'Full Sample'].iloc[0]
    print(f"  Sharpe gap (Low−High): {lv.Sharpe - hv.Sharpe:+.3f} | "
          f"CVaR stress mult: {hv.CVaR95/lv.CVaR95:.2f}x | "
          f"Full-sample Sharpe {fs.Sharpe:.3f} masks Low={lv.Sharpe:.3f}/High={hv.Sharpe:.3f}")


  NVDA  (RF=2.5%)
      Label    N   AnnRet   AnnVol   Sharpe  Sortino    MaxDD   Calmar    VaR95   CVaR95
    Low-Vol  614  100.185   40.449    2.415    4.048  -33.439    2.996   -3.710   -4.685
   High-Vol  614   43.403   59.864    0.683    0.977  -63.695    0.681   -6.171   -8.085
Full Sample 1228   69.432   51.077    1.310    1.959  -66.335    1.047   -4.819   -6.896
  Sharpe gap (Low−High): +1.732 | CVaR stress mult: 1.73x | Full-sample Sharpe 1.310 masks Low=2.415/High=0.683

  JPM  (RF=2.5%)
      Label    N   AnnRet   AnnVol   Sharpe  Sortino    MaxDD   Calmar    VaR95   CVaR95
    Low-Vol  614   25.781   19.199    1.213    1.833  -14.218    1.813   -1.912   -2.531
   High-Vol  614    3.571   41.171    0.026    0.037  -42.888    0.083   -3.738   -5.928
Full Sample 1228   14.137   32.115    0.362    0.518  -43.627    0.324   -2.898   -4.557
  Sharpe gap (Low−High): +1.187 | CVaR stress mult: 2.34x | Full-sample Sharpe 0.362 masks Low=1.213/High=0.026

  PG  (RF=2.5%)
      Labe

### 5b. Regime-Conditional CAPM — Beta Instability

HC3 robust standard errors are mandatory: fat-tailed, regime-switching returns
guarantee non-constant error variance, so plain OLS SEs are badly understated.
Δβ > 0 → assets become **more** market-exposed during stress — precisely when it hurts most.

In [9]:
beta_df = capm.compute_regime_betas(rets_a, regime)

print(f"{'Ticker':<6}  {'Full β':<9} {'Low-Vol β':<11} {'High-Vol β':<12} {'Δβ':<8} {'Δβ%':<8} {'Full R²'}")
print('=' * 68)
for _, row in beta_df.iterrows():
    print(f"{row.Ticker:<6}  {row.Full_beta:>7.3f}   {row.LowVol_beta:>9.3f}   "
          f"{row.HighVol_beta:>10.3f}   {row.dBeta:>6.3f}   {row.dBeta_pct:>5.1f}%  {row.Full_r2:>6.3f}")
    print(f"        95% CI β (full): [{row.Full_blo:.3f}, {row.Full_bhi:.3f}]  p={row.Full_pb:.4f}")

Ticker  Full β    Low-Vol β   High-Vol β   Δβ       Δβ%      Full R²
NVDA      1.742       2.006        1.692   -0.314   -15.6%   0.521
        95% CI β (full): [1.627, 1.857]  p=0.0000
JPM       1.117       0.937        1.149    0.212    22.6%   0.542
        95% CI β (full): [1.011, 1.223]  p=0.0000
PG        0.586       0.434        0.614    0.180    41.4%   0.344
        95% CI β (full): [0.497, 0.676]  p=0.0000


In [10]:
fig = plots.plot_beta_scatter(rets_a, regime, beta_df)
fig.savefig('../data/beta_scatter.png', dpi=150, bbox_inches='tight')

### 5c. Volatility Modelling — EWMA & GARCH(1,1)

Best GARCH error distribution selected by AIC across {normal, t, skewed-t}.
Returns scaled to % internally for GARCH numerical stability.

In [11]:
vol_models = volatility.fit_all_vol_models(rets_a)

for t, m in vol_models.items():
    p = m['garch_fit'].params if m['garch_fit'] else {}
    persist = p.get('alpha[1]', 0) + p.get('beta[1]', 0)
    print(f"{t}: GARCH dist={m['garch_dist']}, "
          f"α+β={persist:.4f}{'  ⚠ IGARCH' if persist >= 1 else ''}  | "
          f"EWMA fcast={m['ewma_fcast']*np.sqrt(252)*100:.2f}%  "
          f"GARCH fcast={m['garch_fcast']*np.sqrt(252)*100:.2f}%")

NVDA: GARCH dist=t, α+β=0.9758  | EWMA fcast=26.44%  GARCH fcast=31.85%
JPM: GARCH dist=t, α+β=0.9762  | EWMA fcast=12.76%  GARCH fcast=15.93%
PG: GARCH dist=skewt, α+β=0.9341  | EWMA fcast=16.16%  GARCH fcast=15.51%
SPY: GARCH dist=skewt, α+β=0.9872  | EWMA fcast=10.12%  GARCH fcast=10.26%


In [12]:
fig = plots.plot_volatility(vol_models)
fig.savefig('../data/volatility.png', dpi=150, bbox_inches='tight')

---
## 6. Evaluation
### 6a. VaR Backtesting — Kupiec POF + Christoffersen Independence

A well-calibrated VaR model must pass **both** tests:
- **POF** (Kupiec 1995): violation frequency ≈ 5%.
- **IND** (Christoffersen 1998): violations not clustered.

`PASS` = fail to reject H₀ at 5% → correct model behaviour.

In [13]:
bt_df = backtest.backtest_var(rets_a, regime)

print(f"{'Ticker':<6} {'Regime':<12} {'n':<6} {'Exceed':<7} {'Rate%':<7} "
      f"{'POF p':<10} {'POF':<6} {'IND p':<10} {'IND'}")
print('=' * 78)
for _, row in bt_df.iterrows():
    pof = 'PASS' if row.pof_pass else 'FAIL'
    ind = 'PASS' if row.ind_pass else 'FAIL'
    print(f"{row.Ticker:<6} {row.Regime:<12} {row['T']:<6} {row.N:<7} "
          f"{row.pi_hat*100:<7.2f} {row.p_pof:<10.4f} {pof:<6} {row.p_ind:<10.4f} {ind}")

Ticker Regime       n      Exceed  Rate%   POF p      POF    IND p      IND
NVDA   Full Sample  1168   74      6.34    0.0439     FAIL   0.0505     PASS
NVDA   Low-Vol      555    22      3.96    0.2460     PASS   0.2145     PASS
NVDA   High-Vol     613    52      8.48    0.0003     FAIL   0.3341     PASS
JPM    Full Sample  1168   61      5.22    0.7289     PASS   0.0035     FAIL
JPM    Low-Vol      555    16      2.88    0.0132     FAIL   0.3319     PASS
JPM    High-Vol     613    45      7.34    0.0126     FAIL   0.0036     FAIL
PG     Full Sample  1168   72      6.16    0.0776     PASS   0.1009     PASS
PG     Low-Vol      555    24      4.32    0.4551     PASS   0.1367     PASS
PG     High-Vol     613    48      7.83    0.0029     FAIL   0.0285     FAIL
SPY    Full Sample  1168   61      5.22    0.7289     PASS   0.0435     FAIL
SPY    Low-Vol      555    19      3.42    0.0714     PASS   0.2763     PASS
SPY    High-Vol     613    42      6.85    0.0459     FAIL   0.0219     FAIL


### 6b. Historical Stress Testing

Equal-weight portfolio (25% each) evaluated during two historically identified stress events.
Weights are parameterised — change `portfolio_weights` to test any allocation.

In [14]:
portfolio_weights = {'NVDA': 0.25, 'JPM': 0.25, 'PG': 0.25, 'SPY': 0.25}

asset_records, portfolio_summary = stress.run_stress_tests(
    rets_a, weights=portfolio_weights
)

asset_df = pd.DataFrame(asset_records)
for name, grp in asset_df.groupby('event'):
    print(f'\n{"="*65}\n  {name}\n{"="*65}')
    print(grp[['ticker','n_days','cum_ret_pct','max_dd_pct','stress_beta']]
          .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

print('\nPortfolio summary:')
for s in portfolio_summary:
    print(f"  {s['name']}: cum={s['p_cum']*100:+.2f}%  "
          f"P&L=${s['dollar_pnl']:+,.0f}  "
          f"daily CVaR={s['p_cvar']*100:.3f}%")


  2020 COVID Crash
ticker  n_days  cum_ret_pct  max_dd_pct  stress_beta
  NVDA      24      -28.240     -37.554          NaN
   JPM      24      -41.735     -42.519          NaN
    PG      24      -21.759     -22.889          NaN
   SPY      24      -33.400     -33.717        1.000

  2022 Rate Shock
ticker  n_days  cum_ret_pct  max_dd_pct  stress_beta
  NVDA     115      -46.935     -48.186        2.370
   JPM     115      -27.404     -32.159        0.853
    PG     115      -17.599     -18.909        0.429
   SPY     115      -22.564     -23.010        1.000

Portfolio summary:
  2020 COVID Crash: cum=-31.68%  P&L=$-316,800  daily CVaR=-11.757%
  2022 Rate Shock: cum=-29.59%  P&L=$-295,876  daily CVaR=-4.094%


---
## 7. Summary Dashboard

In [15]:
fig = plots.plot_dashboard(
    metrics_df=metrics_df,
    regime=regime,
    beta_df=beta_df,
    vol_models=vol_models,
    stress_summary=portfolio_summary,
    save_path='../data/regime_dashboard.png',
)
print('Dashboard saved → data/regime_dashboard.png')

Dashboard saved → data/regime_dashboard.png


In [16]:
# Save the tables quoted in the README so every number traces to a file
out = Path('../outputs'); out.mkdir(exist_ok=True)
metrics_df.to_csv(out / 'regime_metrics.csv', index=False)
beta_df.to_csv(out / 'regime_betas.csv')
bt_df.to_csv(out / 'var_backtest.csv', index=False)
pd.DataFrame(portfolio_summary).to_csv(out / 'stress_portfolio.csv', index=False)
regime_method_df.to_csv(out / 'regime_method_comparison.csv', index=False)
print('Saved tables to outputs/')


Saved tables to outputs/
